# Fig 5a: TIL zones on a TIL-high and a TIL-low Visium section

Produces:
- **Fig 5a**: TIL zones (TIL+, BZ1, BZ2, TIL−) of ER_G_631a_vis (TIL-high) and
  CR_G_310a_vis (TIL-low), coloured as the published panel.
- The H&E view of `plot_visium_til` (TIL zones over the H&E image, ER_G_672a_vis), not a
  paper panel.

Origin (Spencer Watson):
- The published style (dark red / orange / yellow / cadet-blue spots, no H&E, no axes) is the
  "Visualize high and low TIL samples" cell of `Analysis_Repo/Notebooks/TIL_Niche_Comparisons_v2_clean.ipynb`
  (cell 26, `_plot_til_zones_ax`), which drew `TIL_zones_facet.png`.
- `plot_visium_til` and its call on ER_G_672a_vis: `fig6a_0_NicheNet_runs.ipynb`, cells 184–185.

Comparison with the published panel: the published TIL-high and TIL-low maps are crops of
ER_G_631a_vis and CR_G_310a_vis (identified by matching the TIL+ island layout; zones from the
frozen `til_zone_annotations.csv`). The facet cell's automatic selection (top / bottom TIL+
fraction) picks other sections with the current zones (printed below), so the two sections are
set explicitly. The panel shows the whole sections (the published panel is cropped in the
figure layout); the scale bar is 1 mm. ER_G_672a_vis, the section of `plot_visium_til`, is not
the one in the published panel.

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import zarr
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from sklearn.neighbors import NearestNeighbors

from gbmcollaboration import AllDataCenters
from gbmcollaboration.definitions import DATA_PATH_CHUV_FILES, DATA_PATH_CR_FILES, DATA_PATH_UKER_FILES
from gbmcollaboration.paper import config, output

In [ ]:
# ===== USER CONFIG =====
OUT = output.setup("fig5a_TIL_zones_map")
TIL_HIGH_SAMPLE = "ER_G_631a_vis"
TIL_LOW_SAMPLE = "CR_G_310a_vis"
HE_SAMPLE = "ER_G_672a_vis"   # plot_visium_til call of NicheNet_v3
SCALE_BAR_UM = 1000
SPOT_PITCH_UM = 100.0          # Visium centre-to-centre distance

dot_size = 22
spot_linewidth = 0
spot_edgecolor = "none"
panel_width = 4.5
panel_height = 4.5
bg_color = "white"
legend_fontsize = 10
legend_markersize = 8

_ZONE_MAP = {"TIL+": "TIL+", "BZ1": "Border1", "BZ2": "Border2", "TIL-": "TIL−"}
_ZONE_ORDER = ["TIL+", "Border1", "Border2", "TIL−"]
_ZONE_LABELS = {"TIL+": "TIL+", "Border1": "BZ1", "Border2": "BZ2", "TIL−": "TIL-"}
_ZONE_COLORS = {
    "TIL+": "#8B0000",
    "Border1": "#FF8C00",
    "Border2": "#FFD300",
    "TIL−": "#5F9EA0",
}
_OTHER_COLOR = "#D0D0D0"
_ZONE_ZORDER = {"TIL+": 5, "Border1": 4, "Border2": 3, "TIL−": 2}

# TIL zone colors of plot_visium_til
TIL_COLORS = {
    "TIL+": "#E31A1C",   # red - TIL positive core
    "BZ1": "#FF7F00",    # orange - border zone 1 (closer to TIL+)
    "BZ2": "#FFD700",    # yellow - border zone 2 (further from TIL+)
    "TIL-": "#377EB8",   # blue - TIL negative
}

In [ ]:
def _til_percent_by_sample(adata):
    key = "patient" if "patient" in adata.obs.columns else "sample"
    z = adata.obs["TIL_zone"].astype(str)
    df = (
        adata.obs.assign(_til=(z == "TIL+").astype(int))
        .groupby(key, as_index=False)
        .agg(TIL_Positive_Spots=("_til", "sum"), Total_Spots=("_til", "size"))
    )
    df["TIL_Positive_Pct"] = 100.0 * df["TIL_Positive_Spots"] / df["Total_Spots"]
    return df.rename(columns={key: "sample"})


def _plot_til_zones_ax(ax, adata, sample_id):
    ad = adata[adata.obs["sample"] == sample_id].copy()
    if ad.n_obs == 0:
        ax.axis("off")
        return

    coords = ad.obs[["imagecol", "imagerow"]].to_numpy(float)
    m = np.isfinite(coords).all(1)
    if m.sum() == 0:
        ax.axis("off")
        return
    coords = coords[m]
    zones_raw = ad.obs["TIL_zone"].astype(str).values[m]
    zones = np.array([_ZONE_MAP.get(z, z) for z in zones_raw])
    colors = np.array([_ZONE_COLORS.get(z, _OTHER_COLOR) for z in zones])
    zorders = np.array([_ZONE_ZORDER.get(z, 1) for z in zones])

    for zo in sorted(set(zorders)):
        mask = zorders == zo
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=colors[mask], s=dot_size,
            linewidths=spot_linewidth, edgecolors=spot_edgecolor,
            zorder=zo, rasterized=True,
        )

    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(bg_color)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    return coords


def add_scale_bar(ax, coords, length_um=SCALE_BAR_UM):
    """Scale bar in the lower left corner; pixel size from the median spot pitch."""
    d, _ = NearestNeighbors(n_neighbors=2).fit(coords).kneighbors(coords)
    px_per_um = float(np.median(d[:, 1])) / SPOT_PITCH_UM
    x0, y1 = coords[:, 0].min(), coords[:, 1].max()
    ax.plot([x0, x0 + length_um * px_per_um], [y1 + 40 * px_per_um] * 2, color="white", lw=4,
            solid_capstyle="butt", zorder=10)
    ax.plot([x0, x0 + length_um * px_per_um], [y1 + 40 * px_per_um] * 2, color="black", lw=2.5,
            solid_capstyle="butt", zorder=11)


def plot_visium_til(
    sample_ids,
    adata,
    zones=None,
    exclude_til_negative=True,
    img_key="hires",
    figsize_per_panel=(8, 8),
    spot_alpha=0.85,
    he_alpha=1.0,
    show_he_only=False,
    show_legend=True,
    legend_fontsize=9,
    title_fontsize=11,
    save_path=None,
):
    """
    Plot TIL zones on Visium H&E images.

    Parameters
    ----------
    sample_ids : str or list of sample IDs
    adata : AnnData with TIL_zone column in obs
    zones : list of str, optional
        Which zones to show (e.g., ["TIL+", "BZ1", "BZ2", "TIL-"]). If None, shows all.
    exclude_til_negative : bool
        If True (default), don't plot TIL- spots
    show_he_only : bool
        If True, show side-by-side: spots+H&E and H&E only
    """
    if isinstance(sample_ids, str):
        sample_ids = [sample_ids]

    if "TIL_zone" not in adata.obs.columns:
        raise ValueError("TIL_zone not found in adata.obs. Run the TIL annotation loading cell first.")

    # Determine which zones to plot
    available_zones = adata.obs["TIL_zone"].dropna().unique().tolist()
    if zones is None:
        zones = available_zones
    if exclude_til_negative and "TIL-" in zones:
        zones = [z for z in zones if z != "TIL-"]

    # Center mapping
    center_map = {
        "CH": ("CHUV", DATA_PATH_CHUV_FILES),
        "ER": ("UKER", DATA_PATH_UKER_FILES),
        "CR": ("CR", DATA_PATH_CR_FILES),
    }

    # Figure layout
    n_samples = len(sample_ids)
    if show_he_only:
        n_cols = 2
        n_rows = n_samples
        fig_w = figsize_per_panel[0] * 2
        fig_h = figsize_per_panel[1] * n_samples
    else:
        n_cols = min(3, n_samples)
        n_rows = int(np.ceil(n_samples / n_cols))
        fig_w = figsize_per_panel[0] * n_cols
        fig_h = figsize_per_panel[1] * n_rows

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)

    s3 = s3fs.S3FileSystem(anon=False)

    for idx, sample_id in enumerate(sample_ids):
        if show_he_only:
            ax_spots = axes[idx, 0]
            ax_he = axes[idx, 1]
        else:
            ax_spots = axes[idx // n_cols, idx % n_cols]
            ax_he = None

        try:
            # Determine center
            prefix = sample_id[:2]
            if prefix not in center_map:
                raise ValueError(f"Unknown center prefix '{prefix}'")
            center_name, data_paths = center_map[prefix]

            # Find zarr path
            zarr_list = data_paths["Spatial Zarr Files"]
            zarr_path = None
            for p in zarr_list:
                if sample_id in p:
                    zarr_path = p
                    break

            if zarr_path is None:
                raise ValueError(f"No zarr found for {sample_id}")

            # Load image from Zarr
            store = s3.get_mapper(zarr_path)
            root = zarr.open_group(store=store, mode="r")
            table_keys = list(root["tables"].group_keys())
            table_grp = root["tables"]["table"] if "table" in table_keys else root["tables"][table_keys[0]]
            adata_img = ad.read_zarr(table_grp)

            lib = next(iter(adata_img.uns["spatial"].keys()))
            img = adata_img.uns["spatial"][lib]["images"][img_key]
            scalefactors = adata_img.uns["spatial"][lib]["scalefactors"]
            sf_key = "tissue_hires_scalef" if img_key == "hires" else "tissue_lowres_scalef"
            sf = float(scalefactors[sf_key])

            # Coordinates
            if {"pxl_col_in_fullres", "pxl_row_in_fullres"}.issubset(adata_img.obs.columns):
                coords_base = adata_img.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()
            else:
                coords_base = np.asarray(adata_img.obsm["spatial"])

            # Compute scale
            nn = NearestNeighbors(n_neighbors=2).fit(coords_base)
            d, _ = nn.kneighbors(coords_base)
            median_pitch_px = float(np.median(d[:, 1]))

            pitch_um = 100.0
            spot_diam_um = 55.0

            h_px, w_px = img.shape[:2]
            coords_max = coords_base.max(axis=0)
            coords_fullres = (coords_max[0] > 3 * w_px) or (coords_max[1] > 3 * h_px)

            if coords_fullres:
                um_per_px_fullres = pitch_um / median_pitch_px
                um_per_px_img = um_per_px_fullres / sf
                x_um = coords_base[:, 0] * um_per_px_fullres
                y_um = coords_base[:, 1] * um_per_px_fullres
            else:
                um_per_px_img = pitch_um / median_pitch_px
                x_um = coords_base[:, 0] * um_per_px_img
                y_um = coords_base[:, 1] * um_per_px_img

            w_um = w_px * um_per_px_img
            h_um = h_px * um_per_px_img
            img_extent = [0.0, w_um, h_um, 0.0]

            pad_frac = 0.05
            span = max(x_um.max() - x_um.min(), y_um.max() - y_um.min())
            pad_um = pad_frac * span
            x_lim = (x_um.min() - pad_um, x_um.max() + pad_um)
            y_lim = (y_um.min() - pad_um, y_um.max() + pad_um)

            r_um = spot_diam_um / 2.0

            # Get TIL zone for this sample from adata
            sample_mask = adata.obs["sample"].astype(str) == sample_id
            adata_sample = adata[sample_mask]

            # Build barcode -> TIL zone mapping
            # adata barcodes include sample prefix, need to extract just barcode part
            barcode_to_til = {}
            for bc in adata_sample.obs_names:
                bc_str = str(bc)
                til_zone = adata_sample.obs.loc[bc, "TIL_zone"]
                # Extract barcode suffix (after sample_id_)
                if bc_str.startswith(sample_id + "_"):
                    barcode_only = bc_str.replace(sample_id + "_", "")
                else:
                    barcode_only = bc_str
                barcode_to_til[barcode_only] = til_zone

            # Match to image barcodes
            img_barcodes = adata_img.obs_names.astype(str)
            til_values = [barcode_to_til.get(bc, None) for bc in img_barcodes]

            n_spots = len(x_um)

            # Create colors and alphas
            rgba = np.zeros((n_spots, 4))
            for i in range(n_spots):
                til = til_values[i]
                if til in zones and til is not None:
                    rgba[i, :3] = to_rgb(TIL_COLORS.get(til, "#808080"))
                    rgba[i, 3] = spot_alpha
                else:
                    rgba[i, 3] = 0  # transparent

            # Create patches
            patches = [Circle((x_um[i], y_um[i]), r_um) for i in range(n_spots)]
            pc = PatchCollection(patches, linewidth=0.0)
            pc.set_facecolors(rgba)

            # Plot with spots
            ax_spots.imshow(img, extent=img_extent, alpha=he_alpha)
            ax_spots.add_collection(pc)
            ax_spots.set_xlim(x_lim)
            ax_spots.set_ylim(y_lim[1], y_lim[0])
            ax_spots.set_aspect("equal")
            ax_spots.axis("off")
            ax_spots.set_title(f"{sample_id}" + (" - TIL zones" if show_he_only else ""),
                              fontsize=title_fontsize)

            # Plot H&E only if requested
            if ax_he is not None:
                ax_he.imshow(img, extent=img_extent, alpha=1.0)
                ax_he.set_xlim(x_lim)
                ax_he.set_ylim(y_lim[1], y_lim[0])
                ax_he.set_aspect("equal")
                ax_he.axis("off")
                ax_he.set_title(f"{sample_id} - H&E only", fontsize=title_fontsize)

        except Exception as e:
            import traceback
            ax_spots.set_title(f"{sample_id}: ERROR", fontsize=title_fontsize)
            ax_spots.text(0.5, 0.5, str(e)[:80], ha='center', va='center',
                         transform=ax_spots.transAxes, fontsize=8, wrap=True)
            ax_spots.axis("off")
            if ax_he is not None:
                ax_he.axis("off")
            traceback.print_exc()

    # Hide empty panels
    if not show_he_only:
        for idx in range(n_samples, n_rows * n_cols):
            axes[idx // n_cols, idx % n_cols].axis("off")

    # Legend - order zones logically
    if show_legend:
        zone_order = ["TIL+", "BZ1", "BZ2", "TIL-"]
        legend_zones = [z for z in zone_order if z in zones]
        handles = [Line2D([0], [0], marker='o', color='w',
                          markerfacecolor=TIL_COLORS[z], markersize=10,
                          label=z) for z in legend_zones]
        if n_samples == 1 and not show_he_only:
            axes[0, 0].legend(handles=handles, loc='upper right', fontsize=legend_fontsize,
                              frameon=True, framealpha=0.9, edgecolor='none')
        else:
            fig.legend(handles=handles, loc='center right', bbox_to_anchor=(1.08, 0.5),
                       fontsize=legend_fontsize, frameon=False)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Saved: {save_path}")

    return fig, axes

## Load spots and TIL zones

Spot coordinates of the three sections (regular loader, same exclusions as the cohort) and
the frozen TIL zones (`fig5_0_TIL_zones.ipynb`).

In [ ]:
samples = [TIL_HIGH_SAMPLE, TIL_LOW_SAMPLE, HE_SAMPLE]
centres = {"mosaic_chuv": "CH", "mosaic_uker": "ER", "mosaic_cr": "CR"}
loaded = [dc.load_spatial(exclude_low_quality_visium=True, exclude_idh1_mut=True, exclude_patient_ids=True,
                          exclude_relapse=True, list_sample_id=[s for s in samples if s.startswith(centres[name])],
                          load_gex=False, load_deconvolution=True)
          for name, dc in AllDataCenters.data_centers.items() if any(s.startswith(centres[name]) for s in samples)]
spots = pd.concat([o.celltype_df[["imagerow", "imagecol"]] for o in loaded])
spots.index = pd.Index([f"{s}_{b}" for s, b in spots.index], name="spot_id")

til = pd.read_csv(config.TIL_ZONES_CSV, index_col="spot_id")
obs = til.join(spots, how="inner")
obs["TIL_zone"] = pd.Categorical(obs["TIL_zone"], categories=["TIL+", "BZ1", "BZ2", "TIL-"])
adata = ad.AnnData(obs=obs)
print(obs.groupby("sample", observed=True)["TIL_zone"].value_counts().unstack().to_string())

# Sections the facet cell would pick with the current zones (top / bottom TIL+ fraction)
pct = _til_percent_by_sample(ad.AnnData(obs=til))
pct = pct[pct["Total_Spots"] >= 700]
top = pct[pct["TIL_Positive_Spots"].between(6, 800)].sort_values("TIL_Positive_Pct", ascending=False)
bot = pct[pct["TIL_Positive_Spots"] >= 6].sort_values("TIL_Positive_Pct")
print("\nFacet-cell selection: top", top["sample"].head(3).tolist(), "bottom", bot["sample"].head(3).tolist())
print(pct.set_index("sample").loc[[TIL_HIGH_SAMPLE, TIL_LOW_SAMPLE]].round(2).to_string())

## Fig 5a

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(panel_width * 2, panel_height + 0.6))
plot_data = []
for ax, sid, title in zip(axes, [TIL_HIGH_SAMPLE, TIL_LOW_SAMPLE], ["TIL-high", "TIL-low"]):
    coords = _plot_til_zones_ax(ax, adata, sid)
    add_scale_bar(ax, coords)
    ax.set_title(title, fontsize=14)
    plot_data.append(adata.obs.loc[adata.obs["sample"] == sid, ["sample", "imagecol", "imagerow", "TIL_zone"]])
handles = [Line2D([0], [0], marker="o", linestyle="None", color=_ZONE_COLORS[k], label=_ZONE_LABELS[k],
                  markersize=legend_markersize, markeredgewidth=0) for k in _ZONE_ORDER]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=legend_fontsize,
           handletextpad=0.3, columnspacing=1.5)
plt.tight_layout(rect=[0, 0.06, 1, 1])
output.save(OUT, "til_zones_til_high_low", fig, pd.concat(plot_data), panel="Fig 5a")
plt.show()

## TIL zones over the H&E (`plot_visium_til`, NicheNet_v3 cell 185)

In [ ]:
fig, axes = plot_visium_til(
    sample_ids=[HE_SAMPLE],
    adata=adata,
    zones=None,  # None = all available, or e.g. ["TIL+", "BZ1", "BZ2"]
    exclude_til_negative=True,  # True = don't show TIL- spots
    figsize_per_panel=(8, 8),
    spot_alpha=0.8,
    he_alpha=1.0,
    show_he_only=True,  # True = side-by-side spots+H&E and H&E only
    show_legend=True,
)
output.save(OUT, f"til_zones_he_{HE_SAMPLE}", fig)
plt.show()

In [ ]:
print(output.versions(["anndata", "matplotlib", "zarr"]).to_string(index=False))